# Phase 2 — Exploratory Analysis

Understand the return distributions and correlation structure of the 6-asset universe before optimizing (Phase 3) or modeling risk (Phase 5).

Reads `raw_prices.csv` and `log_returns.csv` produced by `src/data.py`. Assumes this notebook is run from the `notebooks/` folder, so paths go up one level — adjust if your CSVs live somewhere else.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["figure.figsize"] = (10, 6)

prices = pd.read_csv("../raw_prices.csv", index_col=0, parse_dates=True)
log_returns = pd.read_csv("../log_returns.csv", index_col=0, parse_dates=True)

prices.head()

## 1. Price history — normalized to $100

Raw prices aren't comparable across assets at very different price levels (GLD vs QQQ). Normalizing to a common starting value of 100 makes relative growth directly comparable.

In [ ]:
normalized = prices / prices.iloc[0] * 100

fig, ax = plt.subplots()
for col in normalized.columns:
    ax.plot(normalized.index, normalized[col], label=col)

ax.set_title("Growth of $100 invested, 2016–2026")
ax.set_ylabel("Value ($)")
ax.legend()
plt.show()

## 2. Return distributions

Histogram of daily log returns per asset, with a normal distribution (same mean/std) overlaid in red for comparison. Where the actual histogram has a taller peak and fatter tails than the red curve, that's visual evidence against the normality assumption parametric VaR relies on.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, col in zip(axes, log_returns.columns):
    r = log_returns[col]
    ax.hist(r, bins=60, density=True, alpha=0.6, label="actual")

    x = np.linspace(r.min(), r.max(), 200)
    ax.plot(x, stats.norm.pdf(x, r.mean(), r.std()), "r--", label="normal fit")

    ax.set_title(col)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3. Distribution summary — skew, kurtosis, normality test

Parametric VaR (Phase 5) assumes normal returns. This table is what confirms or challenges that assumption per asset:

- **skew** — which direction the tail leans
- **excess_kurtosis** — how much fatter the tails are than normal (0 = normal, positive = fat tails)
- **jarque_bera_pvalue** — below 0.05 rejects normality with statistical backing

In [ ]:
def distribution_summary(log_returns: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in log_returns.columns:
        r = log_returns[col]
        jb_stat, jb_pvalue = stats.jarque_bera(r)
        rows.append({
            "ticker": col,
            "ann_mean_%": r.mean() * 252 * 100,
            "ann_vol_%": r.std() * (252 ** 0.5) * 100,
            "skew": stats.skew(r),
            "excess_kurtosis": stats.kurtosis(r),  # Fisher's definition: normal = 0
            "jarque_bera_pvalue": jb_pvalue,
        })
    return pd.DataFrame(rows).set_index("ticker")

summary = distribution_summary(log_returns)
summary

## 4. Correlation matrix

What the Phase 3 optimizer can exploit for diversification — the lower the correlation between two assets, the more risk reduction comes from holding both together.

In [ ]:
corr = log_returns.corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns)
ax.set_yticklabels(corr.columns)

for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black")

ax.set_title("Correlation matrix — daily log returns")
fig.colorbar(im)
plt.tight_layout()
plt.show()

corr

## Notes

_Fill in once run: which assets showed the fattest tails / lowest JB p-values, and which pairs came out least correlated._